# ZK-SNARK Private DAO Voting Prototype
### Google Colab Technical Demo & Reproducibility Pipeline
**Author:** Pragma Mumbai ZK Workshop  
**Stack:** Circom 2 + SnarkJS Groth16 (BN128) + Solidity 0.8.20 + Polygon Amoy Testnet  

**Pipeline Sections:**
1. Setup & Environment Verification
2. Circuit Compilation (Circom 2.2+)
3. Development Trusted Setup (Powers of Tau pot12 & zkey)
4. Proof Generation (Merkle Tree Membership & Blinding)
5. Local Proof Verification (SnarkJS CLI)
6. Hardhat Tests & Optional Deployment (Polygon Amoy)
7. Empirical Performance Benchmark (10 Iterations)

## Section 1: Setup & Environment Verification

In [ ]:
# Check hardware environment
import os
import subprocess

!free -h
!lscpu | grep 'Model name\|CPU(s):'
!node --version
!npm --version

In [ ]:
%%bash
# Download and install native Circom binary (v2.2.1 linux-amd64)
curl -L -o /usr/local/bin/circom https://github.com/iden3/circom/releases/download/v2.2.1/circom-linux-amd64
chmod +x /usr/local/bin/circom
circom --version

# Install SnarkJS globally
npm install -g snarkjs
snarkjs --version

In [ ]:
# Clone the repository and install root dependencies
!rm -rf zk-private-voting
!git clone https://github.com/samudragupto/zk-private-voting.git zk-private-voting
%cd zk-private-voting
!npm ci

## Section 2: Circuit Compilation
Compiles `circuits/voting.circom` to R1CS, WASM, and SYM representations, enforcing a 4-level Merkle tree of voter commitments.

In [ ]:
# Compile circuit using project compilation script
!node scripts/compileCircuit.js

# Display exact constraint breakdown
!snarkjs r1cs info circuits/build/voting.r1cs

## Section 3: Development Trusted Setup Ceremony
Runs the Powers of Tau ceremony (`pot12`, supporting up to 4096 constraints) and generates the Groth16 zkey, verification key, and Solidity verifier (`contracts/Verifier.sol`).

In [ ]:
# Execute Groth16 development setup ceremony
!node scripts/setupGroth16.js

## Section 4: Proof Generation
Constructs a sample 16-leaf Merkle tree of voter commitments, generates witness inputs for Leaf #0, and computes a Groth16 zk-SNARK proof.

In [ ]:
# Generate proof and export formatted Solidity calldata
!node scripts/generateProof.js

## Section 5: Local Proof Verification
Verifies the generated proof locally using the SnarkJS CLI and verification key.

In [ ]:
!snarkjs groth16 verify circuits/build/verification_key.json scripts/generated/public.json scripts/generated/proof.json

## Section 6: Smart Contract Compilation & Unit Tests
Compiles Solidity contracts with Hardhat and runs the test suite (14 passing tests including end-to-end Groth16 proof verification).

In [ ]:
# Compile Solidity smart contracts
!npx hardhat compile

# Execute full test suite
!npx hardhat test

### Optional Deployment to Polygon Amoy Testnet
To deploy on Polygon Amoy, populate your private key and run the cell below. No deployment secrets are required for running tests or proof generation.

In [ ]:
# import os
# os.environ["PRIVATE_KEY"] = "your_wallet_private_key_here"
# os.environ["AMOY_RPC_URL"] = "https://rpc-amoy.polygon.technology"
# os.environ["POLYGONSCAN_API_KEY"] = "your_polygonscan_api_key_here"
# !npx hardhat run scripts/deploy.js --network amoy

## Section 7: Empirical Performance Benchmark
Runs 10 iterations measuring real CPU proof generation time, verification time, and memory usage. Results are written to `benchmark_results.json` and `benchmark_results.md`.

In [ ]:
# Run empirical 10-iteration benchmark
!node scripts/benchmark.js

# Display generated markdown benchmark report
!cat benchmark_results.md